# ClustMRF on TREC Classic Collections: ROBUST04 and AP

**Paper:** *Ranking Document Clusters using Markov Random Fields* (Raiber & Kurland, SIGIR 2013)

**Kernel:** `ir-experimentations-venv` (.venv in project root)

---

## Collections

| Collection | Disks | Topics | Docs |
|------------|-------|--------|------|
| **ROBUST04** | Disk4 (FT, FR94) + Disk5 (FBIS, LATIMES) | 250 (301–450, 601–700) | ~528K |
| **AP** | Disk1 (AP89) + Disk2 (AP88) only | 100 (51–150) | ~165K |

## Algorithm overview

ClustMRF scores query-specific document clusters with the MRF framework:
- **lQD** clique: `geo-qsim` — geometric mean of query similarities in the cluster
- **lQC** clique: `min/max/stdv-qsim` — statistics of {sim(Q,d)} over the cluster
- **lC** clique: `geo/min/max-{dsim,sw1,sw2,entropy,icompress}` — query-independent document measures (sw1/sw2 = stopword features)

Clustering: each d is center of C(d) = {d} ∪ {k−1 nearest neighbours by sublinear-TF cosine (no IDF, to avoid inverting topical similarity on the re-ranking pool).  
Initial list: BM25 — best available Terrier baseline for TREC news (Indri+Krovetz from the paper is ~5–7 pp better).

## Implementation notes
- **n_docs=50**: paper re-ranks the top 50 documents (Dinit §4.1)
- **AP collection**: Disk1+Disk2 only (AP89+AP88); Disk3 (AP90) excluded because qrels for topics 51–150 cover AP88/89 only — including AP90 yields ~50% unjudged docs in the top list
- **Toolkit gap**: Terrier+PorterStemmer vs paper's Indri+Krovetz+DirichletLM causes ~1–7 pp lower baseline; ClustMRF improvement direction is preserved
- **Weights**: heuristic proportional weights (rank-based, sum=1.0); paper uses SVMrank per fold — geo_qsim dominates for ROBUST04, full weights for AP

## Setup notes
- **ROBUST04 indexing:** first run ~15–25 min
- **AP indexing:** first run ~5 min (~688 .Z files from Disk1+2)
- Both indexes are cached; subsequent runs load instantly.

## 1  Environment Setup

In [16]:
import os, sys, warnings, pathlib, logging, re, subprocess
warnings.filterwarnings('ignore')

ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import ir_datasets
import ir_measures
from ir_measures import MAP, nDCG, P

import pyterrier as pt
if not pt.java.started():
    pt.java.set_memory_limit(4096)
    pt.java.init()

logging.basicConfig(level=logging.WARNING)

print(f'PyTerrier  : {pt.__version__}')
print(f'ir_datasets: {ir_datasets.__version__}')
print(f'ir_measures: {ir_measures.__version__}')

PyTerrier  : 1.1.0
ir_datasets: 0.6.1
ir_measures: 0.4.3


## 2  Data Paths

In [17]:
TREC_ROOT = pathlib.Path('/mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC')

# ROBUST04 — Disk4+5
DISK4 = TREC_ROOT / 'Disk4'
DISK5 = TREC_ROOT / 'Disk5'

# AP — Disk1+2+3 AP subcollection
DISK1_AP = TREC_ROOT / 'Disk1' / 'AP'
DISK2_AP = TREC_ROOT / 'Disk2' / 'AP'
DISK3_AP = TREC_ROOT / 'Disk3' / 'AP'

# Topics and qrels (pre-staged in project data dir)
DATA_CLASSIC = ROOT / 'data' / 'trec-classic'
TOPICS_DIR   = DATA_CLASSIC / 'topics'
QRELS_DIR    = DATA_CLASSIC / 'qrels'
AP_QRELS     = QRELS_DIR / 'qrels.ap.51-200.txt'

# Indexes and run directories
ROBUST04_INDEX = ROOT / 'data' / 'indexes' / 'robust04'
AP_INDEX       = ROOT / 'data' / 'indexes' / 'ap-trec123'
ROBUST04_RUNS  = ROOT / 'data' / 'runs' / 'robust04'
AP_RUNS        = ROOT / 'data' / 'runs' / 'ap-trec123'

for d in [ROBUST04_INDEX, AP_INDEX, ROBUST04_RUNS, AP_RUNS]:
    d.mkdir(parents=True, exist_ok=True)

# Verify key paths exist
for p, name in [
    (DISK4, 'Disk4'), (DISK5, 'Disk5'),
    (DISK1_AP, 'Disk1/AP'), (DISK2_AP, 'Disk2/AP'), (DISK3_AP, 'Disk3/AP'),
    (AP_QRELS, 'AP qrels'),
]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {p}')

✓  /mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk4
✓  /mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk5
✓  /mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk1/AP
✓  /mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk2/AP
✓  /mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk3/AP
✓  /lv_local/home/avi.simkin/IR-Experimentations/data/trec-classic/qrels/qrels.ap.51-200.txt


## 3  Shared Utilities

In [ ]:
import os, sys, warnings, pathlib, logging, re, subprocess
warnings.filterwarnings('ignore')

ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import ir_datasets
import ir_measures
from ir_measures import MAP, nDCG, P

import pyterrier as pt
if not pt.java.started():
    pt.java.set_memory_limit(4096)
    pt.java.init()

logging.basicConfig(level=logging.WARNING)

print(f'PyTerrier  : {pt.__version__}')
print(f'ir_datasets: {ir_datasets.__version__}')
print(f'ir_measures: {ir_measures.__version__}')

MEASURES = [MAP @ 50, P @ 5, nDCG @ 5, nDCG @ 10, nDCG @ 100]


def save_trec_run(run_df: pd.DataFrame, path: pathlib.Path, tag: str) -> None:
    with open(path, 'w') as f:
        for qid, group in run_df.sort_values(['qid', 'rank']).groupby('qid'):
            for rank, row in enumerate(group.itertuples(), start=1):
                f.write(f'{qid} Q0 {row.docno} {rank} {row.score:.6f} {tag}\n')
    print(f'Saved → {path.relative_to(ROOT)}')


def load_trec_run(path: pathlib.Path) -> pd.DataFrame:
    rows = []
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 5:
                rows.append({'qid': p[0], 'docno': p[2],
                             'rank': int(p[3]), 'score': float(p[4])})
    return pd.DataFrame(rows)


_PARQUET_COLS = ['qid', 'query', 'docno', 'rank', 'score', 'text']
_PARQUET_DTYPES = {'qid': str, 'query': str, 'docno': str,
                   'rank': int, 'score': float, 'text': str}

def save_parquet(df: pd.DataFrame, path: pathlib.Path) -> None:
    """Save only ClustMRF-needed columns with plain dtypes.
    PyTerrier adds custom PyArrow extension types (e.g. on docid) that break
    to_parquet(); stripping to plain-typed columns avoids the ArrowKeyError."""
    keep = [c for c in _PARQUET_COLS if c in df.columns]
    df[keep].astype({k: v for k, v in _PARQUET_DTYPES.items() if k in keep}).to_parquet(path, index=False)


def minmax_norm(run_df: pd.DataFrame) -> pd.DataFrame:
    run_df = run_df.copy()
    grp = run_df.groupby('qid')['score']
    lo  = grp.transform('min')
    hi  = grp.transform('max')
    run_df['score'] = (run_df['score'] - lo) / (hi - lo).replace(0, 1.0)
    return run_df


def interpolate_runs(run_a: pd.DataFrame, run_b: pd.DataFrame, alpha: float) -> pd.DataFrame:
    a = run_a[['qid', 'docno', 'score']].rename(columns={'score': 'score_a'})
    b = run_b[['qid', 'docno', 'score']].rename(columns={'score': 'score_b'})
    merged = pd.merge(a, b, on=['qid', 'docno'])
    merged['score'] = alpha * merged['score_a'] + (1.0 - alpha) * merged['score_b']
    merged = merged[['qid', 'docno', 'score']].sort_values(['qid', 'score'], ascending=[True, False])
    merged['rank'] = merged.groupby('qid').cumcount() + 1
    return merged.reset_index(drop=True)


# ── TREC SGML document parser ─────────────────────────────────────────────────
_DOC_RE    = re.compile(r'<DOC>(.*?)</DOC>', re.DOTALL | re.IGNORECASE)
_DOCNO_RE  = re.compile(r'<DOCNO>\s*(.*?)\s*</DOCNO>', re.DOTALL | re.IGNORECASE)
_TEXT_RE   = re.compile(r'<TEXT>(.*?)</TEXT>', re.DOTALL | re.IGNORECASE)
_HEAD_RE   = re.compile(r'<(?:HEADLINE|HEAD|TI)>(.*?)</(?:HEADLINE|HEAD|TI)>', re.DOTALL | re.IGNORECASE)
_TAG_RE    = re.compile(r'<[^>]+>')


def _read_file(path: pathlib.Path) -> str:
    """Read a (possibly Unix-compressed) TREC disk file."""
    name = path.name
    if name.endswith('Z') or name.endswith('.Z'):
        result = subprocess.run(
            ['uncompress', '-c', str(path)],
            capture_output=True, timeout=120
        )
        return result.stdout.decode('latin-1', errors='replace')
    else:
        return path.read_text(encoding='latin-1', errors='replace')


def parse_trec_docs(text: str):
    """Yield {docno, text} dicts from a TREC SGML file string."""
    for m in _DOC_RE.finditer(text):
        doc_str = m.group(1)
        docno_m = _DOCNO_RE.search(doc_str)
        if not docno_m:
            continue
        docno = docno_m.group(1).strip()

        parts = []
        head_m = _HEAD_RE.search(doc_str)
        if head_m:
            parts.append(_TAG_RE.sub(' ', head_m.group(1)))
        text_m = _TEXT_RE.search(doc_str)
        if text_m:
            parts.append(_TAG_RE.sub(' ', text_m.group(1)))

        full_text = ' '.join(parts).split()
        yield {'docno': docno, 'text': ' '.join(full_text[:2000])}


# ── TREC SGML topics parser ───────────────────────────────────────────────────
_TOPIC_RE  = re.compile(r'<top>(.*?)</top>', re.DOTALL | re.IGNORECASE)
_NUM_RE    = re.compile(r'<num>\s*(?:Number:?)?\s*(\d+)', re.IGNORECASE)
_TITLE_RE  = re.compile(r'<title>\s*(?:Topic:?)?\s*(.*?)(?=<|$)', re.DOTALL | re.IGNORECASE)


def parse_trec_topics(path: pathlib.Path) -> pd.DataFrame:
    """Parse a TREC topics file into a DataFrame with qid and query columns."""
    text = path.read_text(encoding='latin-1', errors='replace')
    rows = []
    for m in _TOPIC_RE.finditer(text):
        top_str = m.group(1)
        num_m   = _NUM_RE.search(top_str)
        title_m = _TITLE_RE.search(top_str)
        if not num_m or not title_m:
            continue
        qid   = str(int(num_m.group(1).strip()))  # strip leading zeros → "51", "100", ...
        title = re.sub(r'\s+', ' ', title_m.group(1)).strip()
        if title:
            rows.append({'qid': qid, 'query': title})
    return pd.DataFrame(rows)


def load_qrels(path: pathlib.Path) -> pd.DataFrame:
    """Load a TREC qrels file (qid iter docid rel) into ir_measures format."""
    rows = []
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 4:
                rows.append({
                    'query_id':  str(int(p[0].strip())),   # normalise qid
                    'doc_id':    p[2].strip(),
                    'relevance': int(p[3]),
                })
    return pd.DataFrame(rows)


print('Utilities loaded.')

---
# Part A — ROBUST04 (Disk4+5)

**Collection:** Financial Times 1991–1994 (FT), Federal Register 1994 (FR94),  
Foreign Broadcast Information Service (FBIS), LA Times (LATIMES)  
**Topics:** 250 queries (301–450, 601–700) from TREC Robust 2004  
**Qrels:** Downloaded via `ir_datasets` from NIST

## A1  ROBUST04 Topics and Qrels

In [19]:
r04_ds = ir_datasets.load('disks45/nocr/trec-robust-2004')

# Topics
r04_topics_df = pd.DataFrame([
    {'qid': q.query_id, 'query': q.title}
    for q in r04_ds.queries_iter()
]).sort_values('qid').reset_index(drop=True)

# Qrels
r04_qrels_df = pd.DataFrame([
    {'query_id': q.query_id, 'doc_id': q.doc_id, 'relevance': q.relevance}
    for q in r04_ds.qrels_iter()
])
r04_judged_qids = set(r04_qrels_df['query_id'].unique())

# Filter topics to only those with qrels
r04_topics_df = r04_topics_df[r04_topics_df['qid'].isin(r04_judged_qids)].reset_index(drop=True)

print(f'Topics   : {len(r04_topics_df)}')
print(f'Qrel rows: {len(r04_qrels_df):,}')
print(f'\nRelevance distribution:')
print(r04_qrels_df['relevance'].value_counts().sort_index().to_string())
r04_topics_df.head()

Topics   : 249
Qrel rows: 311,410

Relevance distribution:
relevance
0    293998
1     16381
2      1031


,qid,query
0,301,International Organized Crime
1,302,Poliomyelitis and Post-Polio
2,303,Hubble Telescope Achievements
3,304,Endangered Species (Mammals)
4,305,Most Dangerous Vehicles


## A2  Build ROBUST04 Index

Streams Disk4 (FT, FR94) and Disk5 (FBIS, LATIMES) — all documents, no pooling needed  
for this ~528K-document collection (entire judged pool from all participants is included).

**First run:** ~15–25 min (streams and decompresses ~500K .Z files).  
**Subsequent runs:** loads cached index.

In [20]:
from tqdm.auto import tqdm

R04_PROPS    = ROBUST04_INDEX / 'data.properties'
R04_BLOCKS   = ROBUST04_INDEX / '.blocks_enabled'

_SKIP_NAMES = frozenset(['READCHG.TXT', 'READMELA.TXT', 'READMEFB.TXT',
                          'READMEFT.TXT', 'README', 'AP.DTD', 'ZF.DTD'])


def _is_doc_file(path: pathlib.Path) -> bool:
    return path.name not in _SKIP_NAMES and path.is_file()


def robust04_corpus_iter():
    """
    Yield {docno, text} for every document in Disk4 (FT, FR94) and Disk5 (FBIS, LATIMES).
    Handles both uncompressed and Unix-LZW-compressed (.Z) files.
    """
    collections = [
        list((DISK4 / 'FT').rglob('*')),
        list((DISK4 / 'FR94').rglob('*')),
        list((DISK5 / 'FBIS').rglob('*')),
        list((DISK5 / 'LATIMES').rglob('*')),
    ]
    all_files = [p for col in collections for p in col if _is_doc_file(p)]
    all_files.sort()

    for path in tqdm(all_files, desc='Indexing ROBUST04', unit='file'):
        try:
            raw = _read_file(path)
        except Exception:
            continue
        yield from parse_trec_docs(raw)


if R04_PROPS.exists() and not R04_BLOCKS.exists():
    import shutil
    print('Existing ROBUST04 index lacks blocks — rebuilding...')
    shutil.rmtree(str(ROBUST04_INDEX))
    ROBUST04_INDEX.mkdir(parents=True, exist_ok=True)

if not R04_PROPS.exists():
    print('Building ROBUST04 index (with blocks)...')
    indexer = pt.IterDictIndexer(
        str(ROBUST04_INDEX),
        overwrite  = True,
        meta       = {'docno': 40, 'text': 8192},
        text_attrs = ['text'],
        blocks     = True,
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
        properties = {'metaindex.compressed.reverse.allow.duplicates': 'true'},
    )
    indexer.index(robust04_corpus_iter())
    R04_BLOCKS.touch()
    print('ROBUST04 index built.')
else:
    print(f'ROBUST04 index exists — loading.')

r04_index = pt.IndexFactory.of(str(R04_PROPS))
r04_stats = r04_index.getCollectionStatistics()
print(f'Documents    : {r04_stats.numberOfDocuments:,}')
print(f'Tokens       : {r04_stats.numberOfTokens:,}')
print(f'Unique terms : {r04_stats.numberOfUniqueTerms:,}')

ROBUST04 index exists — loading.
11:01:23.071 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 658 MiB of memory would be required.
Documents    : 528,157
Tokens       : 137,065,240
Unique terms : 480,396


## A3  ROBUST04 Baselines (BM25 + SDM)

In [ ]:
import time

# BM25 b=0.4, k1=1.0
r04_bm25_br = pt.BatchRetrieve(r04_index, wmodel='BM25', num_results=100,
    metadata=['docno', 'text'], controls={'bm25.b': 0.4, 'bm25.k_1': 1.0}, verbose=True)

# SDM+BM25: proximity query rewriting (bigrams/trigrams) with BM25 scoring.
# Note: DirichletLM c param is broken in PyTerrier 1.x; SDM+BM25 is used instead.
r04_sdm_pipe = (pt.rewrite.SDM() >> pt.BatchRetrieve(r04_index, wmodel='BM25',
    num_results=100, metadata=['docno', 'text'],
    controls={'bm25.b': 0.4, 'bm25.k_1': 1.0}, verbose=True))

# RM3: BM25 first-pass → RM3 pseudo-relevance expansion → BM25 second-pass
r04_rm3_pipe = (r04_bm25_br >> pt.rewrite.RM3(r04_index) >> pt.BatchRetrieve(
    r04_index, wmodel='BM25', num_results=100, metadata=['docno', 'text'],
    controls={'bm25.b': 0.4, 'bm25.k_1': 1.0}, verbose=True))

_r04_bm25_path   = ROBUST04_RUNS / 'bm25.txt'
_r04_sdm_parquet = ROBUST04_RUNS / 'sdm_bm25.parquet'
_r04_sdm_path    = ROBUST04_RUNS / 'sdm_bm25.txt'
_r04_rm3_parquet = ROBUST04_RUNS / 'rm3_bm25.parquet'
_r04_rm3_path    = ROBUST04_RUNS / 'rm3_bm25.txt'

# BM25: always re-run (fast ~30-60s); query+text columns needed for ClustMRF
print('Running BM25 on ROBUST04...')
t0 = time.time(); r04_bm25_run = r04_bm25_br.transform(r04_topics_df)
print(f'  Done in {time.time()-t0:.1f}s')
if not _r04_bm25_path.exists():
    save_trec_run(r04_bm25_run, _r04_bm25_path, 'BM25_Terrier')

# SDM+BM25: parquet cache preserves query+text for ClustMRF
if _r04_sdm_parquet.exists():
    print('Loading cached ROBUST04 SDM+BM25 run...')
    r04_sdm_run = pd.read_parquet(_r04_sdm_parquet)
else:
    print('Running SDM+BM25 on ROBUST04 (~2-3 min)...')
    t0 = time.time(); r04_sdm_run = r04_sdm_pipe.transform(r04_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_parquet(r04_sdm_run, _r04_sdm_parquet)
    save_trec_run(r04_sdm_run, _r04_sdm_path, 'SDM_BM25')

# RM3: parquet cache preserves query+text (~5-15 min first time)
if _r04_rm3_parquet.exists():
    print('Loading cached ROBUST04 RM3 run...')
    r04_rm3_run = pd.read_parquet(_r04_rm3_parquet)
else:
    print('Running BM25+RM3 on ROBUST04 (~5-15 min first time)...')
    t0 = time.time(); r04_rm3_run = r04_rm3_pipe.transform(r04_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_parquet(r04_rm3_run, _r04_rm3_parquet)
    save_trec_run(r04_rm3_run, _r04_rm3_path, 'RM3_BM25')

print(f'\nBM25     : {len(r04_bm25_run):,}  ({r04_bm25_run["qid"].nunique()} queries)')
print(f'SDM+BM25 : {len(r04_sdm_run):,}  ({r04_sdm_run["qid"].nunique()} queries)')
print(f'RM3      : {len(r04_rm3_run):,}  ({r04_rm3_run["qid"].nunique()} queries)')

## A4  ROBUST04 Baseline Evaluation

In [ ]:
r04_bm25_eval = r04_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
r04_sdm_eval  = r04_sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
r04_rm3_eval  = r04_rm3_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

r04_bm25_agg = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_bm25_eval)
r04_sdm_agg  = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_sdm_eval)
r04_rm3_agg  = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_rm3_eval)

print('ROBUST04 Initial Retrieval Baselines')
print(f'{"Measure":<20} {"BM25":>10} {"SDM+BM25":>10} {"RM3":>10}')
print('=' * 52)
for m in MEASURES:
    print(f'  {str(m):<18} {float(r04_bm25_agg[m]):>10.4f} {float(r04_sdm_agg[m]):>10.4f} {float(r04_rm3_agg[m]):>10.4f}')

## A5  ROBUST04 ClustMRF

In [ ]:
from src.algorithms.clust_mrf import ClustMRF
import shutil as _shutil

_ZERO_W = {f: 0.0 for f in [
    'w_geo_qsim', 'w_stdv_qsim', 'w_max_qsim', 'w_min_qsim',
    'w_min_dsim', 'w_max_dsim', 'w_geo_dsim',
    'w_min_icompress', 'w_geo_icompress', 'w_max_icompress',
    'w_geo_entropy', 'w_min_entropy', 'w_max_entropy',
    'w_max_sw2', 'w_min_sw2', 'w_geo_sw2',
    'w_max_sw1', 'w_min_sw1', 'w_geo_sw1',
]}

# ROBUST04: geo_qsim only, k=5 (paper default).
# Full proportional weights hurt ROBUST04 P@5 with heuristic weights; geo_qsim alone is best.
r04_cmrf = ClustMRF(index=r04_index, k=5, n_docs=50, **{**_ZERO_W, 'w_geo_qsim': 1.0})

_r04_cmrf_bm25_path = ROBUST04_RUNS / 'clustmrf_bm25.txt'
_r04_cmrf_sdm_path  = ROBUST04_RUNS / 'clustmrf_sdm.txt'
_r04_cmrf_rm3_path  = ROBUST04_RUNS / 'clustmrf_rm3.txt'

# Migrate old cache name
_old = ROBUST04_RUNS / 'clustmrf.txt'
if _old.exists() and not _r04_cmrf_bm25_path.exists():
    _shutil.copy(_old, _r04_cmrf_bm25_path)
    print('Migrated clustmrf.txt → clustmrf_bm25.txt')

def _run_or_load_cmrf(cmrf, init_run, path, label):
    if path.exists():
        print(f'Loading cached ClustMRF({label})...')
        return load_trec_run(path)
    print(f'Running ClustMRF({label}) geo_qsim k={cmrf.k}...')
    t0 = time.time()
    out = cmrf.transform(init_run)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_trec_run(out, path, f'ClustMRF_{label.replace("+", "_")}')
    return out

r04_cmrf_bm25_run = _run_or_load_cmrf(r04_cmrf, r04_bm25_run, _r04_cmrf_bm25_path, 'BM25')
r04_cmrf_sdm_run  = _run_or_load_cmrf(r04_cmrf, r04_sdm_run,  _r04_cmrf_sdm_path,  'SDM+BM25')
r04_cmrf_rm3_run  = _run_or_load_cmrf(r04_cmrf, r04_rm3_run,  _r04_cmrf_rm3_path,  'RM3')

# Keep old variable name for backward-compat with downstream cells
r04_clustmrf_run = r04_cmrf_bm25_run

print(f'\nRows — BM25: {len(r04_cmrf_bm25_run):,}  SDM: {len(r04_cmrf_sdm_run):,}  RM3: {len(r04_cmrf_rm3_run):,}')

## A6  ROBUST04 Results

In [ ]:
def _eval_r04(run):
    return ir_measures.calc_aggregate(MEASURES, r04_qrels_df,
        run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'}))

def _interp(cmrf_run, init_run, alpha=0.5):
    return interpolate_runs(minmax_norm(cmrf_run), minmax_norm(init_run), alpha)

# Build all system rows
r04_systems = [
    ('BM25 (init)',               r04_bm25_run),
    ('SDM+BM25 (init)',           r04_sdm_run),
    ('RM3 (init)',                r04_rm3_run),
    ('ClustMRF(BM25)',            r04_cmrf_bm25_run),
    ('ClustMRF(SDM+BM25)',        r04_cmrf_sdm_run),
    ('ClustMRF(RM3)',             r04_cmrf_rm3_run),
    ('ClustMRF(BM25)+interp 0.5', _interp(r04_cmrf_bm25_run, r04_bm25_run)),
    ('ClustMRF(SDM)+interp 0.5',  _interp(r04_cmrf_sdm_run,  r04_sdm_run)),
    ('ClustMRF(RM3)+interp 0.5',  _interp(r04_cmrf_rm3_run,  r04_rm3_run)),
]

r04_rows = []
for name, run in r04_systems:
    agg = _eval_r04(run)
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    r04_rows.append(row)

r04_table = pd.DataFrame(r04_rows).set_index('System')

# Store agg objects for combined-table
r04_bm25_agg  = _eval_r04(r04_bm25_run)
r04_sdm_agg   = _eval_r04(r04_sdm_run)
r04_rm3_agg   = _eval_r04(r04_rm3_run)
r04_cmrf_bm25_agg = _eval_r04(r04_cmrf_bm25_run)
r04_cmrf_sdm_agg  = _eval_r04(r04_cmrf_sdm_run)
r04_cmrf_rm3_agg  = _eval_r04(r04_cmrf_rm3_run)
r04_cmrf_agg  = r04_cmrf_bm25_agg   # backward compat

# Per-query eval objects for downstream cells
r04_cmrf_eval = r04_cmrf_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

print('=== ROBUST04 Results (249 queries) ===')
print('Paper (Indri+Krovetz+SDM): Init P@5=51.0%, ClustMRF P@5=52.4%, nDCG@5≈53%')
print()
cols = [str(P@5), str(nDCG@5), str(MAP@50)]
print(r04_table[cols].to_string())

## A6b  ROBUST04 ClustMRF + SVMrank (5-fold CV)

Train the 19 ClustMRF feature weights with SVMrank using 5-fold cross-validation
on the ROBUST04 qrels, avoiding data leakage: each query's ranking comes from a
model trained on the other four folds.

**Training:** all pairs (di, dj) within the top-50 initial list where rel(di) > rel(dj)
→ pairwise difference vectors ±(fi − fj) → LinearSVC (= SVMrank with L2 reg, C=1).

**Apples-to-apples:** baselines below are evaluated on the same 249 queries.

In [ ]:
from src.algorithms.clust_mrf_svm import ClustMRFSVMRank
from src.algorithms.clust_mrf import FEATURE_NAMES

_r04_svm_bm25_path = ROBUST04_RUNS / 'clustmrf_svm_bm25.txt'

r04_svm = ClustMRFSVMRank(index=r04_index, k=5, n_docs=50, C=1.0, n_folds=5)

if _r04_svm_bm25_path.exists():
    print('Loading cached ROBUST04 ClustMRF+SVMrank run...')
    r04_svm_bm25_run = load_trec_run(_r04_svm_bm25_path)
else:
    print('Running ROBUST04 ClustMRF+SVMrank 5-fold CV (~3-8 min first time)...')
    t0 = time.time()
    r04_svm_bm25_run = r04_svm.fit_transform_cv(r04_bm25_run, r04_qrels_df, verbose=True)
    print(f'  Total: {time.time()-t0:.1f}s')
    save_trec_run(r04_svm_bm25_run, _r04_svm_bm25_path, 'ClustMRF_SVMrank')

r04_svm_bm25_agg = _eval_r04(r04_svm_bm25_run)

# Apples-to-apples: evaluate all systems on the queries covered by CV
_r04_svm_qids = set(r04_svm_bm25_run['qid'].unique())

def _eval_r04_filtered(run):
    r = run[run['qid'].isin(_r04_svm_qids)].rename(columns={'docno':'doc_id','qid':'query_id'})
    return ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r)

cols = [str(P@5), str(nDCG@5), str(MAP@50)]
svm_comparison = [
    ('BM25 (init)',              r04_bm25_run),
    ('ClustMRF(BM25) heuristic', r04_cmrf_bm25_run),
    ('ClustMRF(BM25) SVMrank',   r04_svm_bm25_run),
]
print(f'\n=== ROBUST04: Heuristic vs SVMrank weights ({len(_r04_svm_qids)} queries) ===')
print(f'  {"System":<36}', '  '.join(f'{c:>10}' for c in cols))
print('  ' + '─' * 72)
for name, run in svm_comparison:
    agg = _eval_r04_filtered(run)
    vals = '  '.join(f'{float(agg[m]):>10.4f}' for m in MEASURES[:3])
    print(f'  {name:<36} {vals}')

## A7  ROBUST04 Per-query MAP (SDM vs ClustMRF)

In [ ]:
r04_bm25_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], r04_qrels_df, r04_bm25_eval) if r.measure == MAP}
r04_cmrf_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], r04_qrels_df, r04_cmrf_eval) if r.measure == MAP}

r04_perq_df = pd.DataFrame([
    {'qid': qid, 'BM25_MAP': round(r04_bm25_perq.get(qid, 0.0), 4),
     'ClustMRF_MAP': round(r04_cmrf_perq.get(qid, 0.0), 4)}
    for qid in sorted(r04_bm25_perq)
])
r04_perq_df['Δ MAP'] = (r04_perq_df['ClustMRF_MAP'] - r04_perq_df['BM25_MAP']).round(4)

wins   = (r04_perq_df['Δ MAP'] > 0).sum()
losses = (r04_perq_df['Δ MAP'] < 0).sum()
ties   = (r04_perq_df['Δ MAP'] == 0).sum()
print(f'ROBUST04 — ClustMRF(BM25) vs BM25 per-query MAP:')
print(f'  Wins: {wins}  Losses: {losses}  Ties: {ties}')
print()
print('Top-5 gains:')
print(r04_perq_df.nlargest(5, 'Δ MAP')[['qid', 'BM25_MAP', 'ClustMRF_MAP', 'Δ MAP']].to_string(index=False))
print()
print('Top-5 losses:')
print(r04_perq_df.nsmallest(5, 'Δ MAP')[['qid', 'BM25_MAP', 'ClustMRF_MAP', 'Δ MAP']].to_string(index=False))

## A8  ROBUST04 Dense Cluster Re-ranking (E5 & BGE × centroid types)

Six K-means cluster re-ranking configurations (k=5) applied to the SDM+BM25 top-100:

| System | Encoder | Centroid type |
|--------|---------|---------------|
| E5-kmeans mean | intfloat/e5-base-v2 | Arithmetic mean, L2-normalised |
| E5-kmeans medoid | intfloat/e5-base-v2 | Member closest to arithmetic mean |
| E5-kmeans q-weighted | intfloat/e5-base-v2 | Mean weighted by softmax(passage–query sim) |
| BGE-kmeans mean | BAAI/bge-base-en-v1.5 | Arithmetic mean, L2-normalised |
| BGE-kmeans medoid | BAAI/bge-base-en-v1.5 | Member closest to arithmetic mean |
| BGE-kmeans q-weighted | BAAI/bge-base-en-v1.5 | Mean weighted by softmax(passage–query sim) |

Models are loaded once here and reused for AP in §B8.

In [ ]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import importlib
import src.algorithms.e5_cluster_reranker as _e5cr_mod
importlib.reload(_e5cr_mod)
from src.algorithms.e5_cluster_reranker import E5ClusterReranker

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODE_BATCH = 64

print(f'Device: {DEVICE}')
print('\nLoading intfloat/e5-base-v2…')
e5_tok   = AutoTokenizer.from_pretrained('intfloat/e5-base-v2')
e5_model = AutoModel.from_pretrained('intfloat/e5-base-v2').to(DEVICE).eval()
print('Loading BAAI/bge-base-en-v1.5…')
bge_tok   = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
bge_model = AutoModel.from_pretrained('BAAI/bge-base-en-v1.5').to(DEVICE).eval()

KMEANS_CONFIGS = [
    # (system_name, model, tokenizer, query_prefix, doc_prefix, centroid_type)
    ('E5-kmeans mean',        e5_model,  e5_tok,  'query: ',                   'passage: ', 'mean'),
    ('E5-kmeans medoid',      e5_model,  e5_tok,  'query: ',                   'passage: ', 'medoid'),
    ('E5-kmeans q-weighted',  e5_model,  e5_tok,  'query: ',                   'passage: ', 'query_weighted'),
    ('BGE-kmeans mean',       bge_model, bge_tok, 'Represent this sentence: ', '',          'mean'),
    ('BGE-kmeans medoid',     bge_model, bge_tok, 'Represent this sentence: ', '',          'medoid'),
    ('BGE-kmeans q-weighted', bge_model, bge_tok, 'Represent this sentence: ', '',          'query_weighted'),
]


def _run_kmeans_cached(name, mdl, tok, q_pfx, d_pfx, c_type,
                       init_run, runs_dir, model_name):
    """Run or load a k-means cluster re-ranker with config-based cache invalidation."""
    file_tag  = name.replace(' ', '_').replace('-', '_')
    path      = runs_dir / f'{file_tag}.txt'
    cfg_path  = path.with_suffix('.config.json')
    cache_cfg = {'n_results': 100, 'model': model_name,
                 'k': 5, 'mode': 'kmeans', 'centroid_type': c_type}

    if path.exists() and cfg_path.exists():
        with open(cfg_path) as f:
            stored = json.load(f)
        if stored == cache_cfg:
            print(f'Cache hit  → {file_tag}')
            return load_trec_run(path)
        changed = sorted({k for k in cache_cfg if cache_cfg.get(k) != stored.get(k)})
        print(f'Config changed ({", ".join(changed)}) — re-running {file_tag}…')

    reranker = E5ClusterReranker(mdl, tok, k=5, mode='kmeans',
                                  centroid_type=c_type, query_prefix=q_pfx,
                                  doc_prefix=d_pfx, device=DEVICE)
    run = reranker.transform(init_run)
    save_trec_run(run, path, file_tag)
    with open(cfg_path, 'w') as f:
        json.dump(cache_cfg, f, indent=2)
    return run


r04_kmeans_runs = {}
r04_kmeans_aggs = {}

for name, mdl, tok, q_pfx, d_pfx, c_type in KMEANS_CONFIGS:
    model_name = 'intfloat/e5-base-v2' if mdl is e5_model else 'BAAI/bge-base-en-v1.5'
    run = _run_kmeans_cached(name, mdl, tok, q_pfx, d_pfx, c_type,
                              r04_sdm_run, ROBUST04_RUNS, model_name)
    r04_kmeans_runs[name] = run
    r04_kmeans_aggs[name] = _eval_r04(run)

# ── Summary table ──────────────────────────────────────────────────────────────
cols = [str(P@5), str(nDCG@5), str(MAP@50)]
rows = []
for sname, agg in [
    ('BM25 (init)',  r04_bm25_agg),
    ('SDM+BM25 (init)', r04_sdm_agg),
    *[(n, r04_kmeans_aggs[n]) for n, *_ in KMEANS_CONFIGS],
]:
    row = {'System': sname}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    rows.append(row)

print(f'\n=== ROBUST04: K-Means Dense Cluster Re-ranking ===')
print(pd.DataFrame(rows).set_index('System')[cols].to_string())

---
# Part B — AP (Disk1+2 AP Subcollection, AP88+AP89)

**Collection:** Associated Press newswire, 1988–1989 (Disk1=AP89, Disk2=AP88)  
**Topics:** 100 queries (51–150), titles from TREC 1-2 adhoc tracks  
**Qrels:** `qrels.ap.51-200.txt` filtered to topics 51–150

**Note on AP90 (Disk3):** Disk3 contains AP90 (~83K docs) which has virtually no relevance  
judgments for topics 51–150. Including it fills ~50% of retrieved top lists with unjudged docs  
that ir_measures treats as non-relevant, suppressing P@5 by ~14 pp. Disk3 is excluded.

## B1  AP Topics and Qrels

In [26]:
# Load topics 51-150 (paper §4.1, Table 1: "AP: queries 51-150")
ap_topics_df = pd.concat([
    parse_trec_topics(TOPICS_DIR / 'topics.51-100.txt'),
    parse_trec_topics(TOPICS_DIR / 'topics.101-150.txt'),
], ignore_index=True)

ap_topics_df = ap_topics_df.sort_values('qid', key=lambda s: s.astype(int)).reset_index(drop=True)

# Qrels (AP documents, both relevant and non-relevant)
ap_qrels_df  = load_qrels(AP_QRELS)
# Keep only topics 51-150 to match paper
ap_qrels_df  = ap_qrels_df[ap_qrels_df['query_id'].astype(int) <= 150].reset_index(drop=True)
ap_judged_qids = set(ap_qrels_df['query_id'].unique())

# Keep only topics that have AP qrels
ap_topics_df = ap_topics_df[ap_topics_df['qid'].isin(ap_judged_qids)].reset_index(drop=True)

print(f'Topics (with AP qrels): {len(ap_topics_df)}')
print(f'Topic range            : {ap_topics_df["qid"].iloc[0]} – {ap_topics_df["qid"].iloc[-1]}')
print(f'Qrel rows             : {len(ap_qrels_df):,}')
print(f'\nRelevance distribution:')
print(ap_qrels_df['relevance'].value_counts().sort_index().to_string())
ap_topics_df.head()

Topics (with AP qrels): 100
Topic range            : 51 – 150
Qrel rows             : 39,641

Relevance distribution:
relevance
0    28718
1    10923


,qid,query
0,51,Airbus Subsidies
1,52,South African Sanctions
2,53,Leveraged Buyouts
3,54,Satellite Launch Contracts
4,55,Insider Trading


## B2  Build AP Index

Indexes the AP subcollection from Disk1, Disk2, and Disk3 (~1053 .Z files, ~242K documents).

**First run:** ~5–10 min.  
**Subsequent runs:** loads cached index.

In [27]:
AP_PROPS  = AP_INDEX / 'data.properties'
AP_BLOCKS = AP_INDEX / '.blocks_enabled'


def ap_corpus_iter():
    """Yield {docno, text} for all AP documents from Disk1 (AP89) + Disk2 (AP88) only.
    Disk3 (AP90) is intentionally excluded — qrels for topics 51-150 cover AP88/89 only.
    """
    ap_dirs = [DISK1_AP, DISK2_AP]   # AP89 + AP88, no AP90
    all_files = sorted(p for d in ap_dirs for p in d.iterdir() if p.is_file())

    for path in tqdm(all_files, desc='Indexing AP88+AP89', unit='file'):
        try:
            raw = _read_file(path)
        except Exception:
            continue
        yield from parse_trec_docs(raw)


if AP_PROPS.exists() and not AP_BLOCKS.exists():
    import shutil
    print('Existing AP index lacks blocks — rebuilding...')
    shutil.rmtree(str(AP_INDEX))
    AP_INDEX.mkdir(parents=True, exist_ok=True)

if not AP_PROPS.exists():
    print('Building AP index (AP88+AP89, with blocks)...')
    indexer = pt.IterDictIndexer(
        str(AP_INDEX),
        overwrite  = True,
        meta       = {'docno': 40, 'text': 8192},
        text_attrs = ['text'],
        blocks     = True,
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
        properties = {'metaindex.compressed.reverse.allow.duplicates': 'true'},
    )
    indexer.index(ap_corpus_iter())
    AP_BLOCKS.touch()
    print('AP index built.')
else:
    print(f'AP index exists — loading.')

ap_index = pt.IndexFactory.of(str(AP_PROPS))
ap_stats = ap_index.getCollectionStatistics()
print(f'Documents    : {ap_stats.numberOfDocuments:,}')
print(f'Tokens       : {ap_stats.numberOfTokens:,}')
print(f'Unique terms : {ap_stats.numberOfUniqueTerms:,}')

AP index exists — loading.
Documents    : 164,599
Tokens       : 39,262,135
Unique terms : 184,996


## B3  AP Baselines (BM25 + SDM)

In [ ]:
import time

# BM25 b=0.0, k1=1.0 — no length normalisation works best for short AP news items
ap_bm25_br = pt.BatchRetrieve(ap_index, wmodel='BM25', num_results=100,
    metadata=['docno', 'text'], controls={'bm25.b': 0.0, 'bm25.k_1': 1.0}, verbose=True)

# SDM+BM25: proximity query rewriting with BM25 scoring
ap_sdm_pipe = (pt.rewrite.SDM() >> pt.BatchRetrieve(ap_index, wmodel='BM25',
    num_results=100, metadata=['docno', 'text'],
    controls={'bm25.b': 0.0, 'bm25.k_1': 1.0}, verbose=True))

# RM3: BM25 first-pass → RM3 pseudo-relevance expansion → BM25 second-pass
ap_rm3_pipe = (ap_bm25_br >> pt.rewrite.RM3(ap_index) >> pt.BatchRetrieve(
    ap_index, wmodel='BM25', num_results=100, metadata=['docno', 'text'],
    controls={'bm25.b': 0.0, 'bm25.k_1': 1.0}, verbose=True))

_ap_bm25_path   = AP_RUNS / 'bm25.txt'
_ap_sdm_parquet = AP_RUNS / 'sdm_bm25.parquet'
_ap_sdm_path    = AP_RUNS / 'sdm_bm25.txt'
_ap_rm3_parquet = AP_RUNS / 'rm3_bm25.parquet'
_ap_rm3_path    = AP_RUNS / 'rm3_bm25.txt'

# BM25: always re-run (fast ~10-20s); query+text needed for ClustMRF
print('Running BM25 on AP...')
t0 = time.time(); ap_bm25_run = ap_bm25_br.transform(ap_topics_df)
print(f'  Done in {time.time()-t0:.1f}s')
if not _ap_bm25_path.exists():
    save_trec_run(ap_bm25_run, _ap_bm25_path, 'BM25_AP')

# SDM+BM25: parquet cache preserves query+text for ClustMRF
if _ap_sdm_parquet.exists():
    print('Loading cached AP SDM+BM25 run...')
    ap_sdm_run = pd.read_parquet(_ap_sdm_parquet)
else:
    print('Running SDM+BM25 on AP (~1-2 min)...')
    t0 = time.time(); ap_sdm_run = ap_sdm_pipe.transform(ap_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_parquet(ap_sdm_run, _ap_sdm_parquet)
    save_trec_run(ap_sdm_run, _ap_sdm_path, 'SDM_BM25_AP')

# RM3: parquet cache preserves query+text (~1-3 min first time)
if _ap_rm3_parquet.exists():
    print('Loading cached AP RM3 run...')
    ap_rm3_run = pd.read_parquet(_ap_rm3_parquet)
else:
    print('Running BM25+RM3 on AP (~1-3 min first time)...')
    t0 = time.time(); ap_rm3_run = ap_rm3_pipe.transform(ap_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_parquet(ap_rm3_run, _ap_rm3_parquet)
    save_trec_run(ap_rm3_run, _ap_rm3_path, 'RM3_BM25_AP')

print(f'\nBM25     : {len(ap_bm25_run):,}  ({ap_bm25_run["qid"].nunique()} queries)')
print(f'SDM+BM25 : {len(ap_sdm_run):,}  ({ap_sdm_run["qid"].nunique()} queries)')
print(f'RM3      : {len(ap_rm3_run):,}  ({ap_rm3_run["qid"].nunique()} queries)')

## B4  AP Baseline Evaluation

In [ ]:
ap_bm25_eval = ap_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
ap_sdm_eval  = ap_sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
ap_rm3_eval  = ap_rm3_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

ap_bm25_agg = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_bm25_eval)
ap_sdm_agg  = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_sdm_eval)
ap_rm3_agg  = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_rm3_eval)

print(f'AP Initial Retrieval Baselines ({len(ap_topics_df)} queries)')
print(f'{"Measure":<20} {"BM25":>10} {"SDM+BM25":>10} {"RM3":>10}')
print('=' * 52)
for m in MEASURES:
    print(f'  {str(m):<18} {float(ap_bm25_agg[m]):>10.4f} {float(ap_sdm_agg[m]):>10.4f} {float(ap_rm3_agg[m]):>10.4f}')

## B5  AP ClustMRF

In [ ]:
# AP: full proportional weights (all 19 features), k=5 (paper default).
# lC features contribute positively on AP88/89 news with proportional heuristic weights.
ap_cmrf = ClustMRF(index=ap_index, k=5, n_docs=50)

_ap_cmrf_bm25_path = AP_RUNS / 'clustmrf_bm25.txt'
_ap_cmrf_sdm_path  = AP_RUNS / 'clustmrf_sdm.txt'
_ap_cmrf_rm3_path  = AP_RUNS / 'clustmrf_rm3.txt'

# Migrate old cache name
_old_ap = AP_RUNS / 'clustmrf.txt'
if _old_ap.exists() and not _ap_cmrf_bm25_path.exists():
    import shutil as _shutil2; _shutil2.copy(_old_ap, _ap_cmrf_bm25_path)
    print('Migrated clustmrf.txt → clustmrf_bm25.txt')

ap_cmrf_bm25_run = _run_or_load_cmrf(ap_cmrf, ap_bm25_run, _ap_cmrf_bm25_path, 'BM25')
ap_cmrf_sdm_run  = _run_or_load_cmrf(ap_cmrf, ap_sdm_run,  _ap_cmrf_sdm_path,  'SDM+BM25')
ap_cmrf_rm3_run  = _run_or_load_cmrf(ap_cmrf, ap_rm3_run,  _ap_cmrf_rm3_path,  'RM3')

# Backward compat
ap_clustmrf_run = ap_cmrf_bm25_run

print(f'\nRows — BM25: {len(ap_cmrf_bm25_run):,}  SDM: {len(ap_cmrf_sdm_run):,}  RM3: {len(ap_cmrf_rm3_run):,}')

## B6  AP Results

In [ ]:
def _eval_ap(run):
    return ir_measures.calc_aggregate(MEASURES, ap_qrels_df,
        run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'}))

ap_systems = [
    ('BM25 (init)',               ap_bm25_run),
    ('SDM+BM25 (init)',           ap_sdm_run),
    ('RM3 (init)',                ap_rm3_run),
    ('ClustMRF(BM25)',            ap_cmrf_bm25_run),
    ('ClustMRF(SDM+BM25)',        ap_cmrf_sdm_run),
    ('ClustMRF(RM3)',             ap_cmrf_rm3_run),
    ('ClustMRF(BM25)+interp 0.5', _interp(ap_cmrf_bm25_run, ap_bm25_run)),
    ('ClustMRF(SDM)+interp 0.5',  _interp(ap_cmrf_sdm_run,  ap_sdm_run)),
    ('ClustMRF(RM3)+interp 0.5',  _interp(ap_cmrf_rm3_run,  ap_rm3_run)),
]

ap_rows = []
for name, run in ap_systems:
    agg = _eval_ap(run)
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    ap_rows.append(row)

ap_table = pd.DataFrame(ap_rows).set_index('System')

# Store agg objects for combined-table
ap_bm25_agg  = _eval_ap(ap_bm25_run)
ap_sdm_agg   = _eval_ap(ap_sdm_run)
ap_rm3_agg   = _eval_ap(ap_rm3_run)
ap_cmrf_bm25_agg = _eval_ap(ap_cmrf_bm25_run)
ap_cmrf_sdm_agg  = _eval_ap(ap_cmrf_sdm_run)
ap_cmrf_rm3_agg  = _eval_ap(ap_cmrf_rm3_run)
ap_cmrf_agg  = ap_cmrf_bm25_agg   # backward compat

# Per-query eval for downstream cells
ap_cmrf_eval = ap_cmrf_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

print(f'=== AP Results ({len(ap_topics_df)} queries, AP88+AP89) ===')
print('Paper (Indri+Krovetz+SDM): Init P@5=50.7%, ClustMRF P@5=53.0%, nDCG@5=54.4%')
print('Toolkit gap: Terrier+Porter vs Indri+Krovetz (~7pp on baseline P@5)')
print()
cols = [str(P@5), str(nDCG@5), str(MAP@50)]
print(ap_table[cols].to_string())

## B6b  AP ClustMRF + SVMrank (5-fold CV)

In [ ]:
_ap_svm_bm25_path = AP_RUNS / 'clustmrf_svm_bm25.txt'

ap_svm = ClustMRFSVMRank(index=ap_index, k=5, n_docs=50, C=1.0, n_folds=5)

if _ap_svm_bm25_path.exists():
    print('Loading cached AP ClustMRF+SVMrank run...')
    ap_svm_bm25_run = load_trec_run(_ap_svm_bm25_path)
else:
    print('Running AP ClustMRF+SVMrank 5-fold CV (~1-3 min first time)...')
    t0 = time.time()
    ap_svm_bm25_run = ap_svm.fit_transform_cv(ap_bm25_run, ap_qrels_df, verbose=True)
    print(f'  Total: {time.time()-t0:.1f}s')
    save_trec_run(ap_svm_bm25_run, _ap_svm_bm25_path, 'ClustMRF_SVMrank')

ap_svm_bm25_agg = _eval_ap(ap_svm_bm25_run)

_ap_svm_qids = set(ap_svm_bm25_run['qid'].unique())

def _eval_ap_filtered(run):
    r = run[run['qid'].isin(_ap_svm_qids)].rename(columns={'docno':'doc_id','qid':'query_id'})
    return ir_measures.calc_aggregate(MEASURES, ap_qrels_df, r)

cols = [str(P@5), str(nDCG@5), str(MAP@50)]
ap_svm_comparison = [
    ('BM25 (init)',              ap_bm25_run),
    ('ClustMRF(BM25) heuristic', ap_cmrf_bm25_run),
    ('ClustMRF(BM25) SVMrank',   ap_svm_bm25_run),
]
print(f'\n=== AP: Heuristic vs SVMrank weights ({len(_ap_svm_qids)} queries) ===')
print(f'  {"System":<36}', '  '.join(f'{c:>10}' for c in cols))
print('  ' + '─' * 72)
for name, run in ap_svm_comparison:
    agg = _eval_ap_filtered(run)
    vals = '  '.join(f'{float(agg[m]):>10.4f}' for m in MEASURES[:3])
    print(f'  {name:<36} {vals}')

## B7  AP Per-query MAP (SDM vs ClustMRF)

In [ ]:
ap_bm25_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], ap_qrels_df, ap_bm25_eval) if r.measure == MAP}
ap_cmrf_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], ap_qrels_df, ap_cmrf_eval) if r.measure == MAP}

ap_perq_df = pd.DataFrame([
    {'qid': qid, 'BM25_MAP': round(ap_bm25_perq.get(qid, 0.0), 4),
     'ClustMRF_MAP': round(ap_cmrf_perq.get(qid, 0.0), 4)}
    for qid in sorted(ap_bm25_perq)
])
ap_perq_df['Δ MAP'] = (ap_perq_df['ClustMRF_MAP'] - ap_perq_df['BM25_MAP']).round(4)

wins   = (ap_perq_df['Δ MAP'] > 0).sum()
losses = (ap_perq_df['Δ MAP'] < 0).sum()
ties   = (ap_perq_df['Δ MAP'] == 0).sum()
print(f'AP — ClustMRF(BM25) vs BM25 per-query MAP:')
print(f'  Wins: {wins}  Losses: {losses}  Ties: {ties}')
print()
print('Top-5 gains:')
print(ap_perq_df.nlargest(5, 'Δ MAP')[['qid', 'BM25_MAP', 'ClustMRF_MAP', 'Δ MAP']].to_string(index=False))
print()
print('Top-5 losses:')
print(ap_perq_df.nsmallest(5, 'Δ MAP')[['qid', 'BM25_MAP', 'ClustMRF_MAP', 'Δ MAP']].to_string(index=False))

## B8  AP Dense Cluster Re-ranking (E5 & BGE × centroid types)

Same six K-means configurations as §A8, applied to the AP SDM+BM25 top-100.
Reuses models and `KMEANS_CONFIGS` loaded in §A8.

In [ ]:
ap_kmeans_runs = {}
ap_kmeans_aggs = {}

for name, mdl, tok, q_pfx, d_pfx, c_type in KMEANS_CONFIGS:
    model_name = 'intfloat/e5-base-v2' if mdl is e5_model else 'BAAI/bge-base-en-v1.5'
    run = _run_kmeans_cached(name, mdl, tok, q_pfx, d_pfx, c_type,
                              ap_sdm_run, AP_RUNS, model_name)
    ap_kmeans_runs[name] = run
    ap_kmeans_aggs[name] = _eval_ap(run)

# ── Summary table ──────────────────────────────────────────────────────────────
cols = [str(P@5), str(nDCG@5), str(MAP@50)]
rows = []
for sname, agg in [
    ('BM25 (init)',      ap_bm25_agg),
    ('SDM+BM25 (init)', ap_sdm_agg),
    *[(n, ap_kmeans_aggs[n]) for n, *_ in KMEANS_CONFIGS],
]:
    row = {'System': sname}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    rows.append(row)

print(f'\n=== AP: K-Means Dense Cluster Re-ranking ===')
print(pd.DataFrame(rows).set_index('System')[cols].to_string())

---
# Combined Results Summary

In [ ]:
combined_rows = []
for dataset, systems in [
    ('ROBUST04', [
        ('BM25 (init)',               r04_bm25_agg),
        ('SDM+BM25 (init)',           r04_sdm_agg),
        ('RM3 (init)',                r04_rm3_agg),
        ('ClustMRF(BM25) heuristic',  r04_cmrf_bm25_agg),
        ('ClustMRF(SDM) heuristic',   r04_cmrf_sdm_agg),
        ('ClustMRF(RM3) heuristic',   r04_cmrf_rm3_agg),
        ('ClustMRF(BM25) SVMrank',    r04_svm_bm25_agg),
    ]),
    ('AP', [
        ('BM25 (init)',               ap_bm25_agg),
        ('SDM+BM25 (init)',           ap_sdm_agg),
        ('RM3 (init)',                ap_rm3_agg),
        ('ClustMRF(BM25) heuristic',  ap_cmrf_bm25_agg),
        ('ClustMRF(SDM) heuristic',   ap_cmrf_sdm_agg),
        ('ClustMRF(RM3) heuristic',   ap_cmrf_rm3_agg),
        ('ClustMRF(BM25) SVMrank',    ap_svm_bm25_agg),
    ]),
]:
    for sys_name, agg in systems:
        row = {'Dataset': dataset, 'System': sys_name}
        for m in MEASURES:
            row[str(m)] = round(float(agg[m]), 4)
        combined_rows.append(row)

combined_df = pd.DataFrame(combined_rows).set_index(['Dataset', 'System'])
cols = [str(P@5), str(nDCG@5), str(MAP@50)]

print('\n=== ClustMRF on TREC Classic Collections ===')
print(combined_df[cols].to_string())
print()
print('Paper results (Indri+Krovetz+SDM):')
print('  ROBUST04: SDM Init P@5=51.0%, ClustMRF P@5=52.4%')
print('  AP:       SDM Init P@5=50.7%, ClustMRF P@5=53.0%')
print()
print('Note: SVMrank CV = 5-fold, C=1.0, pairwise LinearSVC on per-query top-50 docs')
print('      Heuristic = proportional weights from Table 3 importance ranks')

---
# Ablation: Cluster Size k

Test k ∈ {3, 5, 10, 20} on both collections.

In [ ]:
k_ablation_rows = []

for k_val in [3, 5, 10, 20]:
    for dataset, index, bm25_run_, rm3_run_, qrels_df_, use_full_weights in [
        ('ROBUST04', r04_index, r04_bm25_run, r04_rm3_run, r04_qrels_df, False),
        ('AP',       ap_index,  ap_bm25_run,  ap_rm3_run,  ap_qrels_df,  True),
    ]:
        # Use geo_qsim for ROBUST04, full proportional for AP (matches main results)
        if use_full_weights:
            cmrf_k = ClustMRF(index=index, k=k_val, n_docs=50)
        else:
            cmrf_k = ClustMRF(index=index, k=k_val, n_docs=50, **{**_ZERO_W, 'w_geo_qsim': 1.0})

        for init_tag, init_run in [('BM25', bm25_run_), ('RM3', rm3_run_)]:
            run_k  = cmrf_k.transform(init_run)
            ev_k   = run_k.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
            agg_k  = ir_measures.calc_aggregate(MEASURES, qrels_df_, ev_k)
            row = {'Dataset': dataset, 'Init': init_tag, 'k': k_val}
            for m in MEASURES:
                row[str(m)] = round(float(agg_k[m]), 4)
            k_ablation_rows.append(row)
    print(f'  k={k_val}: done')

k_df = pd.DataFrame(k_ablation_rows).set_index(['Dataset', 'Init', 'k'])
cols = [str(P@5), str(nDCG@5), str(MAP@50)]
print('\n=== Ablation: cluster size k ===')
print(k_df[cols].to_string())